# Transfer: A Model-Building Checklist

This notebook turns the module into a checklist you can reuse on homework or a new small regression problem.

By the end of this notebook, you should be able to:

- start a regression analysis from a practical question rather than a software menu;
- combine model fitting, diagnostics, unusual-observation checks, and selection carefully;
- write a short regression report that separates evidence, assumptions, and decisions;
- know when a model is not ready to use.

In [ ]:
from lite_setup import ensure_packages
await ensure_packages()

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.stattools import durbin_watson

sales = pd.read_csv(Path('data/territory_sales.csv'))

## Checklist

Use this sequence when you face a new regression problem:

1. Define the response, predictors, units, and intended use.
2. Plot the response against key predictors and inspect predictor relationships.
3. Fit an initial model that is plausible, not just convenient.
4. Interpret coefficients conditionally and check whether signs make practical sense.
5. Check residual assumptions: form, variance, normality if needed, and independence if ordered.
6. Check multicollinearity with correlations and VIF.
7. Check unusual observations with studentized residuals, leverage, Cook's distance, and DFFITS.
8. If using variable selection, treat it as screening and then re-run diagnostics.
9. Check prediction settings for hidden extrapolation.
10. Report the final model with limitations.

In [ ]:
formula = 'Sales ~ Time + MarketPotential + Advertising + Workload + Rating + CompetitorPressure'
model = smf.ols(formula, data=sales).fit()
print(model.summary().tables[1])
print('Adjusted R-squared:', round(model.rsquared_adj, 4))
print('Durbin-Watson:', round(durbin_watson(model.resid), 3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].scatter(model.fittedvalues, model.resid)
axes[0].axhline(0, color='black', linewidth=1)
axes[0].set_xlabel('Fitted')
axes[0].set_ylabel('Residual')
axes[0].set_title('Residuals vs fitted')
infl = model.get_influence().summary_frame()
axes[1].scatter(infl['hat_diag'], infl['student_resid'])
axes[1].axhline(2, color='gray', linestyle='--')
axes[1].axhline(-2, color='gray', linestyle='--')
axes[1].set_xlabel('Leverage')
axes[1].set_ylabel('Studentized residual')
axes[1].set_title('Outlier/leverage map')
plt.tight_layout()

In [ ]:
X = model.model.exog
names = model.model.exog_names
vif = pd.DataFrame({'term': names, 'VIF': [variance_inflation_factor(X, i) for i in range(X.shape[1])]})
vif[vif['term'] != 'Intercept'].sort_values('VIF', ascending=False)

## Report Template

A concise regression report should include:

- the question and data context;
- the fitted model and units;
- coefficient interpretation for the key predictors;
- inference or prediction results that answer the question;
- diagnostics and any unusual observations;
- a clear limitation statement, especially for observational data, extrapolation, and model selection.

Transfer exercise: choose one model from this module and write a five-sentence report using the template above.